In [ ]:
print("kernel alive")

In [2]:
from experiments.robot.libero.gen_config import GenerateConfig
from experiments.robot.openvla_utils import get_action_head, get_processor, get_proprio_projector, get_vla
from prismatic.vla.constants import NUM_ACTIONS_CHUNK, PROPRIO_DIM


# Instantiate config (see class GenerateConfig in experiments/robot/libero/run_libero_eval.py for definitions)
cfg = GenerateConfig(
    pretrained_checkpoint = "openvla_7b_oft_finetuned",
    use_l1_regression = True,
    use_diffusion = False,
    use_film = False,
    num_images_in_input = 2,
    use_proprio = True,
    load_in_8bit = False,
    load_in_4bit = True,
    center_crop = True,
    num_open_loop_steps = NUM_ACTIONS_CHUNK,
    unnorm_key = "libero_spatial_no_noops",
)

# Load OpenVLA-OFT policy and inputs processor
vla  = get_vla(cfg)
processor = get_processor(cfg)

# Load MLP action head to generate continuous actions (via L1 regression)
action_head = get_action_head(cfg, llm_dim=vla.llm_dim)

# Load proprio projector to map proprio to language embedding space
proprio_projector = get_proprio_projector(cfg, llm_dim=vla.llm_dim, proprio_dim=PROPRIO_DIM)

2026-06-25 10:13:11.178647: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-25 10:13:11.474050: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-25 10:13:11.474138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-25 10:13:11.523281: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-25 10:13:11.623045: I tensorflow/core/platform/cpu_feature_guar

Using LIBERO constants:
  NUM_ACTIONS_CHUNK = 8
  ACTION_DIM = 7
  PROPRIO_DIM = 8
  ACTION_PROPRIO_NORMALIZATION_TYPE = bounds_q99
If needed, manually set the correct constants in `prismatic/vla/constants.py`!


2026-06-25 10:13:19.509600: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-25 10:13:19.517585: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Instantiating pretrained VLA policy...


Loading checkpoint shards: 100%|██████████| 4/4 [00:14<00:00,  3.59s/it]


In [3]:
import pickle
from experiments.robot.openvla_utils import get_vla_action

with open("experiments/robot/libero/sample_libero_spatial_observation.pkl", "rb") as file:
    observation = pickle.load(file)

# Generate robot action chunk (sequence of future actions)
actions = get_vla_action(cfg, vla, processor, observation, observation["task_description"], action_head, proprio_projector)
print("Generated action chunk:")
for act in actions:
    print(act)

Generated action chunk:
[0.153 0.034 0.050 -0.004 0.037 0.006 1.008]
[0.246 0.078 0.104 -0.010 0.039 0.005 1.008]
[0.356 0.166 0.136 -0.009 0.044 0.002 1.008]
[0.515 0.254 0.162 -0.003 0.052 0.003 1.008]
[0.665 0.325 0.167 -0.002 0.051 0.006 1.008]
[0.799 0.404 0.148 -0.003 0.026 0.005 1.008]
[0.842 0.455 0.106 -0.003 0.006 0.007 1.008]
[0.842 0.486 0.072 -0.002 0.002 0.013 1.000]


# breakdown version

In [4]:
with open("experiments/robot/libero/sample_libero_spatial_observation.pkl", "rb") as file:
    observation = pickle.load(file)

In [5]:
# inputs
import torch

obs = observation
task_label = observation["task_description"]
noisy_action_projector = None

DEVICE = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

# get_vla_action

In [ ]:
from experiments.robot.openvla_utils import prepare_images_for_vla

all_images = [obs["full_image"]]
if cfg.num_images_in_input > 1:
    all_images.extend([obs[k] for k in obs.keys() if "wrist" in k])
all_images = prepare_images_for_vla(all_images, cfg)

# Extract primary image and additional images
primary_image = all_images.pop(0)

from experiments.robot.openvla_utils import normalize_proprio

# Process proprioception data if used
proprio = None
if cfg.use_proprio:
    proprio = obs["state"]
    proprio_norm_stats = vla.norm_stats[cfg.unnorm_key]["proprio"]
    proprio = normalize_proprio(proprio, proprio_norm_stats)

prompt = f"In: What action should the robot take to {task_label.lower()}?\nOut:"

inputs = processor(prompt, primary_image).to(DEVICE, dtype=torch.bfloat16)

# Process additional wrist images if any
if all_images:
    all_wrist_inputs = [
        processor(prompt, image_wrist).to(DEVICE, dtype=torch.bfloat16) for image_wrist in all_images
    ]
    # Concatenate all images
    primary_pixel_values = inputs["pixel_values"]
    all_wrist_pixel_values = [wrist_inputs["pixel_values"] for wrist_inputs in all_wrist_inputs]
    inputs["pixel_values"] = torch.cat([primary_pixel_values] + all_wrist_pixel_values, dim=1)

# predict action

In [15]:
inputs.keys()

dict_keys(['input_ids', 'attention_mask', 'pixel_values'])

In [18]:
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
pixel_values = inputs["pixel_values"]

IGNORE_INDEX = -100

if not torch.all(input_ids[:, -1] == 29871):
    input_ids = torch.cat(
        (input_ids, torch.unsqueeze(torch.Tensor([29871]).long(), dim=0).to(input_ids.device)), dim=1
    )

# Create fake labels tensor (needed for action mask)
labels = input_ids.clone()
labels[:] = IGNORE_INDEX

# Get number of tokens in prompt (excluding the start token)
NUM_PROMPT_TOKENS = input_ids.shape[-1] - 1  # Subtract action tokens and stop token

# Prepare inputs by adding necessary tokens
input_ids, attention_mask = vla._prepare_input_for_action_prediction(input_ids, attention_mask)

# Update labels tensor for action mask computation later
labels = vla._prepare_labels_for_action_prediction(labels, input_ids)

all_actions_mask = vla._process_action_masks(labels)

In [20]:
# Get input embeddings and action masks
input_embeddings = vla.get_input_embeddings()(input_ids)

# Extract language embeddings
language_embeddings = input_embeddings[~all_actions_mask].reshape(
    input_embeddings.shape[0], -1, input_embeddings.shape[2]
)

# Process vision features
projected_patch_embeddings = vla._process_vision_features(pixel_values, language_embeddings, False)

### TODO this is where ReVIP should take place

# print('using modeling_prismatic.py in openvla-7b-oft-fintuned... folder')
# print('this is where ReVIP should happen')

# Add proprioceptive features if provided
use_proprio = proprio_projector is not None and proprio is not None
if use_proprio:
    proprio = torch.Tensor(proprio).to(projected_patch_embeddings.device, dtype=projected_patch_embeddings.dtype)
    projected_patch_embeddings = vla._process_proprio_features(
        projected_patch_embeddings, proprio, proprio_projector
    )

# Use diffusion if provided, otherwise use regression or discrete prediction
use_diffusion = noisy_action_projector is not None and hasattr(action_head, "noise_scheduler")

# Calculate number of patches (including proprio token and/or diffusion timestep embedding if present)
NUM_PATCHES = vla.vision_backbone.get_num_patches() * vla.vision_backbone.get_num_images_in_input()
if use_proprio:
    NUM_PATCHES += 1
if use_diffusion:
    NUM_PATCHES += 1

In [22]:
normalized_actions, actions_hidden_states = vla._regression_or_discrete_prediction(
    input_embeddings,
    all_actions_mask,
    projected_patch_embeddings,
    attention_mask,
    labels,
    NUM_PATCHES,
    NUM_PROMPT_TOKENS,
    action_head,
)

# normalized_actions_cpu = normalized_actions.detach().to(torch.float32).cpu()
# actions = vla._unnormalize_actions(normalized_actions_cpu, cfg.unnorm_key)
# actions

In [23]:
normalized_actions

array([[0.067, -0.095, 0.056, -0.025, 0.271, 0.153, 1.008],
       [0.178, -0.037, 0.114, -0.080, 0.281, 0.146, 1.008],
       [0.309, 0.076, 0.148, -0.071, 0.307, 0.131, 1.008],
       [0.498, 0.191, 0.176, -0.017, 0.348, 0.134, 1.008],
       [0.676, 0.283, 0.182, -0.002, 0.346, 0.152, 1.008],
       [0.836, 0.387, 0.161, -0.010, 0.213, 0.145, 1.008],
       [0.887, 0.453, 0.116, -0.014, 0.111, 0.160, 1.008],
       [0.887, 0.492, 0.080, -0.001, 0.089, 0.196, 1.000]], dtype=float32)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReViPTSFiLM(nn.Module):
    def __init__(
        self,
        clue_dim=2048,
        token_dim=4096,
        proj_hidden=2048,
        film_hidden=2048,
        gamma_scale=0.1,
        beta_scale=0.1,
    ):
        super().__init__()

        # Proj(.): clue -> token space
        self.clue_proj = nn.Sequential(
            nn.Linear(clue_dim, proj_hidden),
            nn.SiLU(),
            nn.Linear(proj_hidden, token_dim),
        )

        # h(.): z -> [gamma, beta]
        self.film_head = nn.Sequential(
            nn.Linear(token_dim, film_hidden),
            nn.SiLU(),
            nn.Linear(film_hidden, 2 * token_dim),
        )

        # Learnable nonnegative alpha through softplus
        # Start very small so TS-FiLM is near identity at initialization
        self.alpha_raw = nn.Parameter(torch.tensor(-0.43))

        self.gamma_scale = gamma_scale
        self.beta_scale = beta_scale

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.zeros_(self.film_head[-1].bias)
        nn.init.xavier_uniform_(self.film_head[-1].weight, gain=0.01)

    def forward(self, patch_tokens, clue, return_aux=False):
        """
        patch_tokens: [B, N, D]
        clue:         [B, clue_dim]
        """
        if patch_tokens.dim() != 3:
            raise ValueError(f"patch_tokens must be [B, N, D], got {patch_tokens.shape}")
        if clue.dim() != 2:
            raise ValueError(f"clue must be [B, clue_dim], got {clue.shape}")

        B, N, D = patch_tokens.shape
        if clue.shape[0] != B:
            raise ValueError("Batch size mismatch between patch_tokens and clue")

        if D != self.film_head[-1].out_features // 2:
            raise ValueError(
                f"Token dim mismatch: patch_tokens has D={D}, "
                f"but TS-FiLM expects {self.film_head[-1].out_features // 2}"
            )

        z = self.clue_proj(clue)                    # [B, D]
        gamma_beta = self.film_head(z)             # [B, 2D]
        gamma, beta = gamma_beta.chunk(2, dim=-1)  # [B, D], [B, D]

        # Keep FiLM modulation bounded
        gamma = self.gamma_scale * torch.tanh(gamma)
        beta = self.beta_scale * torch.tanh(beta)

        alpha = F.softplus(self.alpha_raw)         # scalar >= 0

        delta = gamma[:, None, :] * patch_tokens + beta[:, None, :]
        mod_patch_tokens = patch_tokens + alpha * delta

        if return_aux:
            return mod_patch_tokens, {
                "z": z,
                "gamma": gamma,
                "beta": beta,
                "alpha": alpha,
            }

        return mod_patch_tokens
    
import torch
import torch.nn.functional as F


def revip_action_loss(
    pred_actions: torch.Tensor,
    gt_actions: torch.Tensor,
    loss_type: str = "l1",
):
    """
    Loss on generated actions vs ground-truth actions.

    Args:
        pred_actions: [B, ...]
        gt_actions:   [B, ...] same shape as pred_actions
        loss_type:    "l1", "mse", or "smooth_l1"

    Returns:
        loss: scalar tensor
        stats: dict
    """
    if pred_actions.shape != gt_actions.shape:
        raise ValueError(
            f"Shape mismatch: pred_actions {pred_actions.shape} vs gt_actions {gt_actions.shape}"
        )

    if loss_type == "l1":
        loss = F.l1_loss(pred_actions, gt_actions)
    elif loss_type == "mse":
        loss = F.mse_loss(pred_actions, gt_actions)
    elif loss_type == "smooth_l1":
        loss = F.smooth_l1_loss(pred_actions, gt_actions)
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    stats = {
        "loss": loss.detach(),
        "pred_mean_abs": pred_actions.abs().mean().detach(),
        "gt_mean_abs": gt_actions.abs().mean().detach(),
        "error_mean_abs": (pred_actions - gt_actions).abs().mean().detach(),
    }

    return loss, stats

In [ ]:
# path
import os

data_dir = "/home/s2fli/libero/LIBERO-plus/datasets"
spatial_dir = "libero_spatial"
libero_spatial_dir = os.path.join(data_dir, spatial_dir)
tso_path = "/home/s2fli/libero/libero_plus_camparam_rlds/tso_embeddings/libero_spatial"
vla_embedding_path = "/mnt/f/vla_embeddings/libero_spatial"

tse = ReViPTSFiLM().to("cuda", dtype=torch.bfloat16)
optimizer = torch.optim.AdamW(
    tse.parameters(),
    lr=1e-5,
    weight_decay=1e-5,
)

In [ ]:
projected_patch_embeddings

In [ ]:
i = 0 # task
j = 0 # demo
k = 0 # frame

task_file_name = task_label.replace(" ", "_") + "_demo.hdf5"
task_path = os.path.join(libero_spatial_dir, task_file_name)

tso_file = os.path.join(tso_path, "task0_demo0.pt")

tso = torch.load(tso_file, map_location="cuda")

In [ ]:
chunck_size = tso['chunk_size']
tse_embeddings = tse(projected_patch_embeddings, tso['chunk_features'][k//chunck_size].to(torch.bfloat16).reshape(1, -1))
projected_patch_embeddings = tse_embeddings

In [ ]:
# Add proprioceptive features if provided
use_proprio = proprio_projector is not None and proprio is not None
if use_proprio:
    proprio = torch.Tensor(proprio).to(projected_patch_embeddings.device, dtype=projected_patch_embeddings.dtype)
    projected_patch_embeddings = vla._process_proprio_features(
        projected_patch_embeddings, proprio, proprio_projector
    )

# Use diffusion if provided, otherwise use regression or discrete prediction
use_diffusion = noisy_action_projector is not None and hasattr(action_head, "noise_scheduler")

# Calculate number of patches (including proprio token and/or diffusion timestep embedding if present)
NUM_PATCHES = vla.vision_backbone.get_num_patches() * vla.vision_backbone.get_num_images_in_input()
if use_proprio:
    NUM_PATCHES += 1
if use_diffusion:
    NUM_PATCHES += 1

In [ ]:
normalized_actions, actions_hidden_states = vla._regression_or_discrete_prediction_train(
    input_embeddings,
    all_actions_mask,
    projected_patch_embeddings,
    attention_mask,
    labels,
    NUM_PATCHES,
    NUM_PROMPT_TOKENS,
    action_head,
)

In [ ]:
import torch
import numpy as np
from prismatic.vla.constants import ACTION_PROPRIO_NORMALIZATION_TYPE, NormalizationType

def _unnormalize_actions(vla, normalized_actions, unnorm_key=None):
    """Unnormalize actions using dataset statistics.

    Accepts either a torch.Tensor or numpy.ndarray as input.
    Returns the same type family as the input when practical.
    """
    input_was_numpy = isinstance(normalized_actions, np.ndarray)
    if input_was_numpy:
        normalized_actions = torch.from_numpy(normalized_actions)

    action_norm_stats = vla.get_action_stats(unnorm_key)

    if ACTION_PROPRIO_NORMALIZATION_TYPE == NormalizationType.BOUNDS:
        ref = action_norm_stats["min"]
        mask = action_norm_stats.get("mask", np.ones_like(ref, dtype=bool))
        action_high = action_norm_stats["max"]
        action_low = action_norm_stats["min"]

    elif ACTION_PROPRIO_NORMALIZATION_TYPE == NormalizationType.BOUNDS_Q99:
        ref = action_norm_stats["q01"]
        mask = action_norm_stats.get("mask", np.ones_like(ref, dtype=bool))
        action_high = action_norm_stats["q99"]
        action_low = action_norm_stats["q01"]

    else:
        raise ValueError("Unsupported action/proprio normalization type detected!")

    mask = torch.as_tensor(mask, device=normalized_actions.device, dtype=torch.bool)
    action_high = torch.as_tensor(
        action_high, device=normalized_actions.device, dtype=normalized_actions.dtype
    )
    action_low = torch.as_tensor(
        action_low, device=normalized_actions.device, dtype=normalized_actions.dtype
    )

    actions = torch.where(
        mask,
        0.5 * (normalized_actions + 1) * (action_high - action_low + 1e-8) + action_low,
        normalized_actions,
    )

    if input_was_numpy:
        return actions.numpy()
    return actions

In [ ]:
action_tso = _unnormalize_actions(vla, normalized_actions, cfg.unnorm_key)

In [ ]:
for p in vla.parameters():
    p.requires_grad_(False)

for p in tse.parameters():
    p.requires_grad_(True)

for p in action_head.parameters():
    p.requires_grad_(False)

for p in proprio_projector.parameters():
    p.requires_grad_(False)

vla.eval()
action_head.eval()
proprio_projector.eval()

print("freeze all VLA parameters")

In [ ]:
action_gt = torch.as_tensor(
    actions,
    device=action_tso.device,
    dtype=action_tso.dtype,
)

In [ ]:
loss, stats = revip_action_loss(action_tso, action_gt)
optimizer.zero_grad()
loss.backward()

In [ ]:
loss